In [1]:
from sklearn import preprocessing
from sklearn.metrics import mean_absolute_percentage_error, r2_score, mean_absolute_percentage_error


# For 2D analysis
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from scipy.optimize import curve_fit
from sklearn.preprocessing import MinMaxScaler
# from utils import period2freq, freq2period

# For PCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.optimize import curve_fit

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import csv
import time
import glob
import os

# Datasize in KB
data_size_kb = {'4mb': 4096, '16mb': 16384, '64mb': 65536,
            '256mb': 262144, '512mb': 524288, '1gb': 1048576,
            '5gb': 5242880, '50gb': 52428800, '100gb': 104857600,
            '300gb': 314572800,}

# Key Parameters
IOR_PARAMS = ['operation', 'randomOffset', 'transferSize', 
            'aggregateFilesizeMB', 'numTasks', 'totalTime', 
            'numNodes', 'tasksPerNode', 'trMiB', "storageType"]

TARGET_PARAMS = [ "bestStorage" ]
op_dict = {0: "write", 1: "read"}

test_configs = {
    "ddmd_2n_s": {
        "SCRIPT_ORDER": "ddmd_script_order",
        "NUM_NODES": 2,
        "exp_data_path": "./ddmd",
        "test_folders": ['ddmd_2n_pfs_small']
    },
    "ddmd_4n_l": {
        "SCRIPT_ORDER": "ddmd_script_order",
        "NUM_NODES": 4,
        "exp_data_path": "./ddmd",
        "test_folders": ['ddmd_4n_pfs_large']
    },
    "1kg": {
        "SCRIPT_ORDER": "1kg_script_order",
        "NUM_NODES": 2,
        "exp_data_path": "./1kgenome/fastflow_tests",
        "test_folders": ['par_6000_10n_nfs_ps300']
    },
    "pyflex_240f": {
        "SCRIPT_ORDER": "pyflextrkr_script_order",
        "NUM_NODES": 8,
        "exp_data_path": "./pyflextrkr",
        "test_folders": ['summer_sam_8n_pfs']
    },
}

# Load experiment data
curr_test="1kg" # ddmd_2n_s, ddmd_4n_l, 1kg, pyflex_240f

SCRIPT_ORDER = test_configs[curr_test]["SCRIPT_ORDER"]
NUM_NODES = test_configs[curr_test]["NUM_NODES"]
exp_data_path = test_configs[curr_test]["exp_data_path"]
test_folders = test_configs[curr_test]["test_folders"]

# test_folders = ['par_3000_1n_pfs_ps300', 'par_6000_1n_pfs_ps300', 
#                 'par_9000_1n_pfs_ps300'] par_3000_10n_shm_ps300

In [2]:
# My utility functions
import utils.perf_visualize as pv

# Parameter Notes for Datalife:
Each entry in the table represent only one single edge in the workflow. An directed edge connects a **fileName** and a **taskName**, representing data access.

---
- **operation**: The type of I/O operation {0: "write", 1: "read"}, value 1 represents read (e.g. a directed edge edge from a **fileName** to a **taskName**), value 0 represents write (e.g. a directed edge edge from a **taskName** to a **fileName**)
- **randomOffset**: The type of data access pattern { 0: "sequential file access", 1: "random file access"}
- **transferSize**: Average I/O size of the particular I/O operation to a file, calculated from aggregateFilesizeMB/opCount
- **aggregateFilesizeMB**: Total I/O size of a particular I/O operation to a file for a task
- **numTasks**: Number of parallel tasks for this particular task
- **totalTime**: The total I/O time of of a particular I/O operation to a file for a task
- **numNodes**: Number of nodes used for this particular task
- **tasksPerNode**: numTasks/numNodes for a task
- **bwMiB**: transferRate of a particular I/O operation to a file for a task, calculated from aggregateFilesizeMB/totalTime
- **storageType**: The storage type used in this task. {0: "localssd", 1: "beegfs/pfs", 2: "lustre", 3: "unknown"}
- **opCount**: the number of I/O operation count of a particular I/O operation to a file for a task
- **taskName**: the task name that is running for a particular workflow
- **taskPID**: the task PID
- **fileName**: the name of file that a I/O operation is for

In [3]:
def transform_store_code(storage_type):
    if storage_type == "localssd":
        store_code = 0
    elif storage_type == "beegfs" or storage_type == "pfs":
        store_code = 1
    elif storage_type == "lustre":
        store_code = 2
    else:
        store_code = 3
    return store_code

def decode_store_code(store_code):
    if store_code == 0:
        storage_type = "localssd"
    elif store_code == 1:
        storage_type = "beegfs"
    elif store_code == 2:
        storage_type = "lustre"
    else:
        storage_type = "unknown"
    return storage_type



In [4]:
def merge_by_test_param(ssd_df, beegfs_df):
    # Compare and find the row with only storage_type and trMiB columns different but others are the same
    # Merge the dataframes on columns other than 'storageType' and 'trMiB'
    merge_columns = ['operation', 'randomOffset', 'transferSize', 'aggregateFilesizeMB', 'numTasks', 'numNodes', 'tasksPerNode', 'opCount'] # 'totalTime', 
    merged_df = pd.merge(ssd_df, beegfs_df, on=merge_columns, suffixes=('_ssd', '_beegfs'), how='outer')

    # Remove columns storageType_ssd and storageType_beegfs
    merged_df.drop(['storageType_ssd', 'storageType_beegfs'], axis=1, inplace=True)
    
    # # Check if cloumens 'operation_ssd' and 'operation_beegfs' has the same values, if yes combine to 1 column 'operation'
    # merged_df['operation'] = merged_df.apply(lambda row: 0 if row['operation_ssd'] == row['operation_beegfs'] else -1, axis=1)

    # Compare 'trMiB' values and add 'selectStorage' column
    merged_df['selectStorage'] = merged_df.apply(lambda row: 0 if row['trMiB_ssd'] > row['trMiB_beegfs'] else 1, axis=1)

    # Debug: Print unique values of 'operation' column before and after merge
    print("Unique 'operation' values in ssd_df:", ssd_df['operation'].unique())
    print("Unique 'operation' values in beegfs_df:", beegfs_df['operation'].unique())
    print("Unique 'operation' values in merged_df:", merged_df['operation'].unique())
    
    print(merged_df.head(5))
    # print shaoe
    print(f"merged_df.shape: {merged_df.shape}")
    print(f"ssd_df.shape: {ssd_df.shape}")
    print(f"beegfs_df.shape: {beegfs_df.shape}")

    return merged_df




In [5]:
def is_sequential(numbers):
    if not numbers:  # Check if the list is empty
        return False

    sorted_numbers = sorted(numbers)  # Sort the numbers
    return all(sorted_numbers[i] + 1 == sorted_numbers[i + 1] for i in range(len(sorted_numbers) - 1))


def get_stat_file_pids(all_files):
    # Extract target tasks from blk_files
    target_tasks = set()
    for blk_file in all_files:
        # Get the filename without the path
        filename = os.path.basename(blk_file)
        # repalce ".local" for now
        filename = filename.replace(".local", "")
        
        # Split filename by '.'
        parts = filename.split('.')
        # print(f"get_stat_file_pids() : parts = {parts}")
        if len(parts) >= 3:
            # Get the target task from the -3 extension
            task = parts[-3]
            target_tasks.add(task)
    target_tasks = sorted(target_tasks)
    return target_tasks

def add_stat_to_df(trial_folder, monitor_timer_stat_io, 
                   operation, fname, task_pid, store_code):
    
    fname = fname.replace(".local", ".")
    # fileName removed the last 4 extensions
    fileName = ".".join(fname.split(".")[:-4])
    # fileName keep only the basename
    fileName = os.path.basename(fileName)
                        
    # Get write statistics
    # print(f"monitor_timer_stat_write = {monitor_timer_stat_write}")
    tmp_write_stat = {}
    tmp_write_stat['aggregateFilesizeMB'] = pv.file_size_to_mb(monitor_timer_stat_io[2])
    
    # # Below values can only be updated once task name and per task parallelism is known
    # tmp_write_stat['numTasks'] = numTasksWrite
    # if numTasksWrite < numNodes:
    #     tmp_write_stat['numNodes'] = numTasksWrite # **number of used nodes
    # else:
    #     tmp_write_stat['numNodes'] = numNodes
    # tmp_write_stat['tasksPerNode'] = math.ceil(tmp_write_stat['numTasks']/tmp_write_stat['numNodes'])
    # tmp_write_stat['taskName'] = "unknown" #task_name
    
    tmp_write_stat['transferSize'] = monitor_timer_stat_io[2]/monitor_timer_stat_io[1]
    tmp_write_stat['operation'] = int(operation)
    tmp_write_stat['totalTime'] = monitor_timer_stat_io[0]
    tmp_write_stat['trMiB'] = pv.file_size_to_mb(monitor_timer_stat_io[2]/monitor_timer_stat_io[0])
    tmp_write_stat['storageType'] = store_code
    tmp_write_stat['opCount'] = monitor_timer_stat_io[1]
    tmp_write_stat['taskPID'] = task_pid
    tmp_write_stat['fileName'] = fileName
    if tmp_write_stat['totalTime'] > 100:
        print(f"Recorded large totalTime[{monitor_timer_stat_io}] from task_pid[{task_pid}] fileName[{fileName}]")
    # if w_fname == "":
    #     print(f"Write file not found for task_pid[{task_pid}] but has write stat [{tmp_write_stat}]")
    
    op = "w"
    if operation == 1: op = "r"

    # find the w_blk_trace_jsons files with the current task_pid
    w_blk_trace_jsons = glob.glob(f"{trial_folder}/*.{task_pid}.{op}_blk_trace.json")
    write_pattern = 0 # 0: seq, 1: rand        
    for w_blk_trace_json in w_blk_trace_jsons:
        with open(w_blk_trace_json) as f:
            w_blk_trace_data = json.load(f)
            # print(w_blk_trace_data)
            blk_list = w_blk_trace_data['io_blk_range']
            if blk_list[3] == -2:
                write_pattern = 1
                break
    # FIXME: For now only sequential read and write IOR
    tmp_write_stat['randomOffset'] = write_pattern
    
    return tmp_write_stat
    

# TODO: find task PID's input and output to match script name
def get_wf_result_df(tests, wf_params, target_tasks, storageType="localssd"):
    wf_df = pd.DataFrame(columns=wf_params)

    # Identify trial folders
    wf_trial_folders = [
        folder for folder in glob.glob(f"{tests}/*")
        if folder.endswith(("t1", "t2", "t3"))
    ]
    print(f"Trial folders: {wf_trial_folders}")

    store_code = transform_store_code(storageType)

    for trial_folder in wf_trial_folders:
        blk_files = glob.glob(f"{trial_folder}/*_blk_trace.json")
        datalife_jsons = glob.glob(f"{trial_folder}/*.datalife.json")
        target_tasks = get_stat_file_pids(blk_files)

        for datalife_json in datalife_jsons:
            task_pid = os.path.basename(datalife_json).split(".")[1]
            if task_pid not in target_tasks:
                continue

            try:
                with open(datalife_json) as f:
                    datalife_data = json.load(f)
            except json.JSONDecodeError:
                print(f"Error loading file: {datalife_json}")
                continue

            task_name = list(datalife_data.keys())[0]
            monitor_timer_stat = datalife_data[task_name]['monitor']
            system_timer_stat = datalife_data[task_name]['system']
            monitor_timer_targets = ["read", "write"]

            for fname in [f for f in blk_files if f".{task_pid}." in f]:
                op_type = "read" if ".r_blk_trace." in fname else "write"
                monitor_stat = monitor_timer_stat[op_type]

                if pv.file_size_to_mb(monitor_stat[2]) == 0:
                    print(f"No {op_type} stat for task_name[{task_name}] task_pid[{task_pid}]")
                    continue
                
                # taskParallelism = task_name_to_parallelism[task_name]
                       
                tmp_stat = add_stat_to_df(
                    trial_folder, monitor_stat, 
                    1 if op_type == "read" else 0,
                    fname, task_pid, store_code
                )
                wf_df = wf_df._append(tmp_stat, ignore_index=True)

    return wf_df

In [6]:



# Key Parameters
wf_params = ['operation', 'randomOffset', 'transferSize', 
            'aggregateFilesizeMB', 'numTasks', 'totalTime', 
            'numNodes', 'tasksPerNode', 'trMiB', 'storageType',
            'opCount','taskName','taskPID', 'fileName', 'stageOrder']
target_tasks = ["python"] # omit srun from 1kgenome run

all_wf_df = pd.DataFrame(columns=wf_params)

def get_test_folder_dfs(test_folder, wf_params, target_tasks,
                        storageType="localssd", workflow="1kg"):
    folder_dfs = pd.DataFrame(columns=wf_params)

    for tests in test_folder:
        stat_path = f"{exp_data_path}/{tests}"

        # Generate workflow data
        wf_df = get_wf_result_df(stat_path, wf_params, target_tasks,
                                 storageType=storageType)
        print(wf_df.head(5))
        print(f"df shape: {wf_df.shape}")

        # Append workflow data to the folder dataframe
        folder_dfs = folder_dfs._append(wf_df, ignore_index=True)
        
    return folder_dfs

wf_pfs_df = pd.DataFrame(columns=wf_params)



wf_pfs_df = wf_pfs_df._append(get_test_folder_dfs(test_folders, 
                                        wf_params, target_tasks,
                                        storageType="pfs", workflow="ddmd"), ignore_index=True)



Trial folders: ['./1kgenome/fastflow_tests/par_6000_10n_nfs_ps300/300_p_10n_PFS_t1']


/tmp/ipykernel_747884/4256101406.py:134: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  wf_df = wf_df._append(tmp_stat, ignore_index=True)


  operation randomOffset   transferSize  aggregateFilesizeMB numTasks  \
0         0            0   81791.000000             0.078002      NaN   
1         1            0    8191.921474          2421.758036      NaN   
2         1            0    8191.921474          2421.758036      NaN   
3         1            0    8191.912230          2421.794365      NaN   
4         0            0  101056.000000             0.096375      NaN   

   totalTime numNodes tasksPerNode        trMiB storageType opCount taskName  \
0   0.000049      NaN          NaN  1593.503085           1       1      NaN   
1  19.632952      NaN          NaN   123.351699           1  309988      NaN   
2  19.632952      NaN          NaN   123.351699           1  309988      NaN   
3  19.658824      NaN          NaN   123.191215           1  309993      NaN   
4   0.000050      NaN          NaN  1917.557288           1       1      NaN   

       taskPID                fileName stageOrder  
0   6818-dc111  chr2n-2201-2

/tmp/ipykernel_747884/4193535247.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  folder_dfs = folder_dfs._append(wf_df, ignore_index=True)
/tmp/ipykernel_747884/4193535247.py:32: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  wf_pfs_df = wf_pfs_df._append(get_test_folder_dfs(test_folders,


In [7]:
print(wf_pfs_df.head(5))
print(wf_pfs_df.shape)

  operation randomOffset   transferSize  aggregateFilesizeMB numTasks  \
0         0            0   81791.000000             0.078002      NaN   
1         1            0    8191.921474          2421.758036      NaN   
2         1            0    8191.921474          2421.758036      NaN   
3         1            0    8191.912230          2421.794365      NaN   
4         0            0  101056.000000             0.096375      NaN   

   totalTime numNodes tasksPerNode        trMiB storageType opCount taskName  \
0   0.000049      NaN          NaN  1593.503085           1       1      NaN   
1  19.632952      NaN          NaN   123.351699           1  309988      NaN   
2  19.632952      NaN          NaN   123.351699           1  309988      NaN   
3  19.658824      NaN          NaN   123.191215           1  309993      NaN   
4   0.000050      NaN          NaN  1917.557288           1       1      NaN   

       taskPID                fileName stageOrder  
0   6818-dc111  chr2n-2201-2

In [8]:
def match_script_name(tests):
    # Find folders ending with [t1, t2, t3] in the test_folders
    test_folders = glob.glob(f"{tests}/*")
    wf_trial_folders = [folder for folder in test_folders if folder.endswith("t1") or folder.endswith("t2") or folder.endswith("t3")]
    print(f"Trial folders: {wf_trial_folders}")

    pid_input_output_dict = {}

    for trial_folder in wf_trial_folders:
        blk_files = glob.glob(f"{trial_folder}/*_blk_trace.json")
        print(f"len(blk_files) = {len(blk_files)}")
        unique_pids = get_stat_file_pids(blk_files)

        for pid in unique_pids:
            if pid not in pid_input_output_dict:
                pid_input_output_dict[pid] = {
                    "input": [],
                    "output": [],
                    "prevTask": "",
                    "taskName": ""
                }

            # Find the blk_trace_jsons files with the current task_pid
            w_blk_trace_jsons = glob.glob(f"{trial_folder}/*.{pid}.local.w_blk_trace.json")
            r_blk_trace_jsons = glob.glob(f"{trial_folder}/*.{pid}.local.r_blk_trace.json")

            # Replace ".local" with an empty string
            w_blk_trace_jsons = [f.replace(".local", "") for f in w_blk_trace_jsons]
            r_blk_trace_jsons = [f.replace(".local", "") for f in r_blk_trace_jsons]

            # Process write (output) files
            for w_file_path in w_blk_trace_jsons:
                w_file_name_parts = w_file_path.split(".")
                w_file_name = '.'.join(w_file_name_parts[:-3])  # Remove the last 3 extensions
                w_file_basename = os.path.basename(w_file_name)
                pid_input_output_dict[pid]['output'].append(w_file_basename)

            # Process read (input) files
            for r_file_path in r_blk_trace_jsons:
                r_file_name_parts = r_file_path.split(".")
                r_file_name = '.'.join(r_file_name_parts[:-3])  # Remove the last 3 extensions
                r_file_basename = os.path.basename(r_file_name)
                if r_file_basename not in pid_input_output_dict[pid]['input']:
                    pid_input_output_dict[pid]['input'].append(r_file_basename)

    return pid_input_output_dict

def get_wf_pid_script_dict(test_folder):

    all_wf_dict= {}

    for tests in test_folder:
        # check of test folder starts with seq or par
        if tests.startswith("seq"):
            numTasksWrite = 1
            numTasksRead = 1
        else:
            numTasksWrite = 1
            numTasksRead = 1

        # io_size_dfs
        wf_dict = match_script_name(f"{exp_data_path}/{tests}")

        # # corr_matrix(wf_df, storageType)
        all_wf_dict.update(wf_dict)
    return all_wf_dict


all_wf_dict = get_wf_pid_script_dict(test_folders)

print(all_wf_dict)



Trial folders: ['./1kgenome/fastflow_tests/par_6000_10n_nfs_ps300/300_p_10n_PFS_t1']
len(blk_files) = 1810
{'13757-dc026': {'input': ['columns.txt', 'ALL.chr1.250000.vcf'], 'output': ['chr1n-1801-2001.tar.gz'], 'prevTask': '', 'taskName': ''}, '13765-dc026': {'input': ['ALL.chr1.250000.vcf', 'columns.txt'], 'output': ['chr1n-5401-5601.tar.gz'], 'prevTask': '', 'taskName': ''}, '13895-dc026': {'input': ['ALL.chr1.250000.vcf', 'columns.txt'], 'output': ['chr1n-1201-1401.tar.gz'], 'prevTask': '', 'taskName': ''}, '13912-dc026': {'input': ['columns.txt', 'ALL.chr1.250000.vcf'], 'output': ['chr1n-2601-2801.tar.gz'], 'prevTask': '', 'taskName': ''}, '13926-dc026': {'input': ['ALL.chr1.250000.vcf', 'columns.txt'], 'output': ['chr1n-2001-2201.tar.gz'], 'prevTask': '', 'taskName': ''}, '13927-dc026': {'input': ['columns.txt', 'ALL.chr1.250000.vcf'], 'output': ['chr1n-1601-1801.tar.gz'], 'prevTask': '', 'taskName': ''}, '13937-dc026': {'input': ['columns.txt', 'ALL.chr1.250000.vcf'], 'output': [

In [9]:
# Add prevTask column
wf_pfs_df['prevTask'] = ""
wf_pfs_df['taskName'] = "unknown"
print(wf_pfs_df.head(5))
print(wf_pfs_df.shape)

  operation randomOffset   transferSize  aggregateFilesizeMB numTasks  \
0         0            0   81791.000000             0.078002      NaN   
1         1            0    8191.921474          2421.758036      NaN   
2         1            0    8191.921474          2421.758036      NaN   
3         1            0    8191.912230          2421.794365      NaN   
4         0            0  101056.000000             0.096375      NaN   

   totalTime numNodes tasksPerNode        trMiB storageType opCount taskName  \
0   0.000049      NaN          NaN  1593.503085           1       1  unknown   
1  19.632952      NaN          NaN   123.351699           1  309988  unknown   
2  19.632952      NaN          NaN   123.351699           1  309988  unknown   
3  19.658824      NaN          NaN   123.191215           1  309993  unknown   
4   0.000050      NaN          NaN  1917.557288           1       1  unknown   

       taskPID                fileName stageOrder prevTask  
0   6818-dc111  chr

In [10]:
print(wf_pfs_df)
# save to initial df
wf_pfs_df.to_csv(f'first_df.csv', index=False)

     operation randomOffset   transferSize  aggregateFilesizeMB numTasks  \
0            0            0   81791.000000             0.078002      NaN   
1            1            0    8191.921474          2421.758036      NaN   
2            1            0    8191.921474          2421.758036      NaN   
3            1            0    8191.912230          2421.794365      NaN   
4            0            0  101056.000000             0.096375      NaN   
...        ...          ...            ...                  ...      ...   
1805         1            0    1496.071162             4.591329      NaN   
1806         1            0    1209.815295             3.560534      NaN   
1807         1            0    1209.815295             3.560534      NaN   
1808         1            0    1209.815295             3.560534      NaN   
1809         0            0    3521.413347            10.115144      NaN   

      totalTime numNodes tasksPerNode        trMiB storageType opCount  \
0      0.0000

In [11]:
import re
        
def matches_pattern(file_path, patterns):
    """Match a file path against task definition patterns."""
    file_name = os.path.basename(file_path)
    for pattern in patterns:
        try:
            regex_pattern = re.compile(pattern)
            if regex_pattern.fullmatch(file_name):
                return True
        except re.error as e:
            print(f"Invalid regex: {pattern}, Error: {e}")
    return False

def assign_task_names(tasks, task_order_dict):
    """Assign task names and predecessors to tasks based on patterns."""
    for task_pid, details in tasks.items():
        input_paths = details.get('input', [])
        output_paths = details.get('output', [])
        task_name = details.get('taskName', 'unknown')  # Use existing or default to 'unknown'

        # Iterate through each task definition
        for task, definition in task_order_dict.items():
            # Check if any output matches
            if any(matches_pattern(op, definition['outputs']) for op in output_paths):
                task_name = task
                tasks[task_pid]['taskName'] = task_name
                tasks[task_pid]['stage_order'] = definition['stage_order']
                break

            # If no output matches, check for input matches
            for prevTask, predecessor_def in definition['predecessors'].items():
                if any(matches_pattern(ip, predecessor_def.get('inputs', [])) for ip in input_paths):
                    task_name = task
                    tasks[task_pid]['taskName'] = task_name
                    tasks[task_pid]['stage_order'] = definition['stage_order']
                    tasks[task_pid]['prevTask'] = prevTask
                    # print(f"Input match found: Task [{task}] prevTask [{prevTask}] with input_patterns {predecessor_def.get('inputs', [])}")
                    break

        # If no valid match, warn about the task
        if task_name == 'unknown':
            print(f"Warning: Task PID {task_pid} could not be assigned a valid taskName.")

    return tasks
        
# Load task ordering json file
task_order_dict = {}
with open(f"{exp_data_path}/{SCRIPT_ORDER}.json") as f:
    task_order_dict = json.load(f)

print(task_order_dict)
# Create a mapping from taskName to parallelism
task_name_to_parallelism = {task: info['parallelism'] for task, info in task_order_dict.items()}
print(task_name_to_parallelism)

# Fill in task names
assign_task_names(all_wf_dict, task_order_dict)
# Unique list of taskNames
taskNames = set([v['taskName'] for v in all_wf_dict.values()])
print(f"Unique taskNames: {taskNames}")
print(f"all_wf_dict:")
for k,v in all_wf_dict.items():
    print(f"{k}:{v}")
print(f"all_wf_dict-----")

# wf_pfs_df =  assign_task_names(wf_pfs_df, task_order_dict)
# print(wf_pfs_df.head(5))
print(wf_pfs_df['fileName'].unique())
print(wf_pfs_df['taskName'].unique())

{'individuals': {'stage_order': 0, 'parallelism': 300, 'predecessors': {'initial_data': {'inputs': ['ALL\\.chr.*\\.250000\\.vcf', 'columns\\.txt']}}, 'outputs': ['chr.*n-.*-.*\\.tar\\.gz']}, 'individuals_merge': {'stage_order': 1, 'parallelism': 10, 'predecessors': {'individuals': {'inputs': ['chr.*n-.*-.*\\.tar\\.gz']}}, 'outputs': ['chr.*n\\.tar\\.gz']}, 'sifting': {'stage_order': 1, 'parallelism': 10, 'predecessors': {'initial_data': {'inputs': ['ALL\\.chr([1-9]|10)\\.phase3_shapeit2_mvncall_integrated_v5\\.20130502\\.sites\\.annotation\\.vcf']}}, 'outputs': ['sifted.*\\.txt']}, 'mutation_overlap': {'stage_order': 2, 'parallelism': 10, 'predecessors': {'initial_data': {'inputs': ['SAS', 'EAS', 'GBR', 'AMR', 'AFR', 'EUR', 'ALL', 'columns\\.txt']}, 'sifting': {'inputs': ['sifted.*\\.txt']}, 'individuals_merge': {'inputs': ['chr.*n\\.tar\\.gz']}}, 'outputs': ['chr\\d+-[A-Z]{3}\\.tar\\.gz$', 'chr\\d+-[A-Z]{3}\\.txt\\.tar\\.gz$']}, 'frequency': {'stage_order': 2, 'parallelism': 10, 'pred

In [12]:
# Create a mapping from taskPID to taskName
task_pid_to_name = {pid: info['taskName'] for pid, info in all_wf_dict.items()}
# print(task_pid_to_name)
# Update the DataFrame with the taskName
wf_pfs_df['taskName'] = wf_pfs_df['taskPID'].map(task_pid_to_name).fillna('unknown')

# Create a mapping from taskPID to prevTask
task_pid_to_prev_task = {pid: info['prevTask'] for pid, info in all_wf_dict.items()}
# print(task_pid_to_prev_task)
# add prevTask column to the DataFrame
wf_pfs_df['prevTask'] = wf_pfs_df['taskPID'].map(task_pid_to_prev_task).fillna('unknown')

# Print the updated DataFrame
# print(wf_pfs_df.head(5))
print(wf_pfs_df.shape)
df_unknown = wf_pfs_df[wf_pfs_df['taskName'] == 'unknown']
print(f"df unknown ({df_unknown.shape}):\n{df_unknown.head(5)}")
# print(f"df found:\n{wf_pfs_df[wf_pfs_df['taskName'] != 'unknown']}")

for pid, info in all_wf_dict.items():
    if 'stage_order' not in info:
        print(f"Missing 'stage_order' for taskPID: {pid}, info: {info}")

# Create a mapping from taskPID to stage_order
task_pid_to_stage_order = {pid: info['stage_order'] for pid, info in all_wf_dict.items()}
# print(task_pid_to_stage_order)
# add prevTask column to the DataFrame
wf_pfs_df['stageOrder'] = wf_pfs_df['taskPID'].map(task_pid_to_stage_order).fillna('-1')




# # FIXME: temporary 1k genome data fix
# # remove dataframe rows with all_wf_df['taskName'] == 'unknown'
# wf_pfs_df = wf_pfs_df[wf_pfs_df['taskName'] != 'unknown']


# remove rows with filename contaiing string "SIFT.chr*.vcf"
for chrom in range(0, 11):
    wf_pfs_df = wf_pfs_df[~wf_pfs_df['fileName'].str.contains(f"SIFT.chr{chrom}.vcf")]
    
# Adjust dataframe prevTask
for index, row in wf_pfs_df.iterrows():
    if row['operation'] == 0:
        if row['taskName'] == '':
            # Update taskName for write tasks to "" (empty string)
            wf_pfs_df.at[index, 'taskName'] = 'none'
    else:
        # Adjust read task predecessors
        taskName = row['taskName']
        fileName = row['fileName']
        task_definition = task_order_dict[taskName]
        for task, inputs in task_definition['predecessors'].items():
            input_patterns = inputs['inputs']
            if matches_pattern(fileName, input_patterns):
                wf_pfs_df.at[index, 'prevTask'] = task



(1810, 16)
df unknown ((0, 16)):
Empty DataFrame
Columns: [operation, randomOffset, transferSize, aggregateFilesizeMB, numTasks, totalTime, numNodes, tasksPerNode, trMiB, storageType, opCount, taskName, taskPID, fileName, stageOrder, prevTask]
Index: []


In [13]:
print(wf_pfs_df)

     operation randomOffset   transferSize  aggregateFilesizeMB numTasks  \
0            0            0   81791.000000             0.078002      NaN   
1            1            0    8191.921474          2421.758036      NaN   
2            1            0    8191.921474          2421.758036      NaN   
3            1            0    8191.912230          2421.794365      NaN   
4            0            0  101056.000000             0.096375      NaN   
...        ...          ...            ...                  ...      ...   
1805         1            0    1496.071162             4.591329      NaN   
1806         1            0    1209.815295             3.560534      NaN   
1807         1            0    1209.815295             3.560534      NaN   
1808         1            0    1209.815295             3.560534      NaN   
1809         0            0    3521.413347            10.115144      NaN   

      totalTime numNodes tasksPerNode        trMiB storageType opCount  \
0      0.0000

In [14]:
# print(f"df found:\n{wf_pfs_df[wf_pfs_df['taskName'] == 'trackstats']}")


# # Below values can only be updated once task name and per task parallelism is known
import math

# Assuming 'wf_pfs_df' is the DataFrame with a 'taskName' column
for index, row in wf_pfs_df.iterrows():
    task_name = row['taskName']
    if task_name in task_name_to_parallelism:
        task_parallelism = task_name_to_parallelism[task_name]

        # Determine numNodes
        if task_parallelism < NUM_NODES:
            row['numNodes'] = task_parallelism  # Number of used nodes
        else:
            row['numNodes'] = NUM_NODES

        # Update numTasks and tasksPerNode
        row['numTasks'] = task_parallelism
        row['tasksPerNode'] = math.ceil(task_parallelism / NUM_NODES)

        # Update the DataFrame
        wf_pfs_df.at[index, 'numNodes'] = row['numNodes']
        wf_pfs_df.at[index, 'numTasks'] = row['numTasks']
        wf_pfs_df.at[index, 'tasksPerNode'] = row['tasksPerNode']


In [15]:
# Modify numTasks by mapping the "parallelism" from task_order_dict based on taskName

wf_pfs_df['numTasks'] = wf_pfs_df['taskName'].map(task_name_to_parallelism).fillna(1)
# # Update the tasksPerNode column
# wf_pfs_df['tasksPerNode'] = wf_pfs_df['numTasks'] / wf_pfs_df['numNodes']



In [16]:
# Save the updated DataFrame to a CSV file
wf_pfs_df.to_csv(f'{test_folders[0]}.csv', index=False)


In [17]:
# Calculate I/O time per taskName
task_io_time_total = wf_pfs_df.groupby('taskName')['totalTime'].sum()

task_io_time_adjust = {}
total_wf_io_time = 0

print("Total I/O time per stage:")
for task, io_time in task_io_time_total.items():
    # Adjust I/O time by parallelism
    io_time_adjusted = io_time / task_name_to_parallelism[task]
    task_io_time_adjust[task] = io_time_adjusted
    total_wf_io_time+=io_time_adjusted
    
    print(f" {task}: {io_time_adjusted} (sec)")

# print(task_io_time_adjust)
print(f"Total I/O time per workflow: {total_wf_io_time}")

Total I/O time per stage:
 frequency: 0.3303839877 (sec)
 individuals: 39.60488580273667 (sec)
 individuals_merge: 0.4102728527 (sec)
 mutation_overlap: 0.031660314 (sec)
 sifting: 9.8428153928 (sec)
Total I/O time per workflow: 50.220018349936666


In [18]:
# Read from file "./master_ior_df.csv"
df_ior = pd.read_csv("./master_ior_df.csv")
# df_ior = pd.read_csv(f'{test_folders[0]}.csv')
print(df_ior.columns)
print(df_ior.shape)

# oscache size is 25GiB
oscacheSizeMB = 25 * 1024  # Convert to MiB

Index(['Unnamed: 0', 'operation', 'randomOffset', 'transferSize',
       'aggregateFilesizeMB', 'numTasks', 'totalTime_ssd', 'numNodes',
       'tasksPerNode', 'trMiB_ssd', 'trMiB_ave_ssd', 'trMiB_ave_std_ssd',
       'trMiB_ave_std_perc_ssd', 'aggregateFilesizeMB_log_ssd',
       'transferSize_log_ssd', 'aggregateFilesizeMB_log_norm_ssd',
       'trMiB_norm_ssd', 'operation_beegfs', 'totalTime_beegfs',
       'trMiB_beegfs', 'trMiB_ave_beegfs', 'trMiB_ave_std_beegfs',
       'trMiB_ave_std_perc_beegfs', 'aggregateFilesizeMB_log_beegfs',
       'transferSize_log_beegfs', 'aggregateFilesizeMB_log_norm_beegfs',
       'trMiB_norm_beegfs', 'selectStorage', 'RWR_ssd', 'RWR_beegfs'],
      dtype='object')
(4788, 30)


In [ ]:
# Function to calculate transferRate based on bounds
def calculate_transferRate_storage(df_ior, parallelism, transfer_size, par_col, transferRate_column):        
    # Sort df_ior by tasksPerNode and transferSize
    df_ior_sorted = df_ior.sort_values(by=[par_col, 'transferSize'])
    
    # Find lower and upper bounds for tasksPerNode
    lower_tasks = df_ior_sorted[df_ior_sorted[par_col] <= parallelism]
    upper_tasks = df_ior_sorted[df_ior_sorted[par_col] >= parallelism]

    # Take lowest/highest task df if no lower/higher bound found
    if lower_tasks.empty:
        lower_tasks = df_ior_sorted[df_ior_sorted[par_col] == df_ior_sorted[par_col].min()]
    if upper_tasks.empty:
        upper_tasks = df_ior_sorted[df_ior_sorted[par_col] == df_ior_sorted[par_col].max()]
    
    low_bound_tasks = lower_tasks.iloc[-1]
    high_bound_tasks = upper_tasks.iloc[0]
    
    # Filter df_ior by tasksPerNode bounds
    df_ior_within_bounds = df_ior_sorted[
        (df_ior_sorted[par_col] >= low_bound_tasks[par_col]) &
        (df_ior_sorted[par_col] <= high_bound_tasks[par_col])
    ]
    
    if df_ior_within_bounds.empty:
        raise ValueError("No rows found within tasksPerNode bounds.")
    
    # Sort within bounds by transferSize
    df_ior_within_bounds = df_ior_within_bounds.sort_values(by='transferSize')
    
    # Check if there are enough rows to find transferSize bounds
    low_bound_transfer = df_ior_within_bounds[df_ior_within_bounds['transferSize'] < transfer_size]
    high_bound_transfer = df_ior_within_bounds[df_ior_within_bounds['transferSize'] > transfer_size]
    
    # Take lowest/highest transferSize in df if no lower/higher bound found
    if low_bound_transfer.empty:
        low_bound_transfer = df_ior_within_bounds[df_ior_within_bounds['transferSize'] == df_ior_within_bounds['transferSize'].min()]
    if high_bound_transfer.empty:
        high_bound_transfer = df_ior_within_bounds[df_ior_within_bounds['transferSize'] == df_ior_within_bounds['transferSize'].max()]
    
    low_bound = low_bound_transfer.iloc[-1]
    high_bound = high_bound_transfer.iloc[0]
    
    # Perform linear interpolation to estimate the transferRate
    low_size = low_bound['transferSize']
    high_size = high_bound['transferSize']
    low_tr = low_bound[transferRate_column]
    high_tr = high_bound[transferRate_column]
    
    if (high_size - low_size) == 0:
        estimated_tr = low_tr
    else:
        estimated_tr = low_tr + (transfer_size - low_size) * (high_tr - low_tr) / (high_size - low_size)
    return float(estimated_tr)


# Initialize a list to store the updates
updates = []

# Iterate through each row of wf_pfs_df
for index, row in wf_pfs_df.iterrows():
    # Find matching rows in df_ior based on operation
    conditions = (df_ior['operation'] == row['operation'])
    df_ior_matched = df_ior[conditions]

    if not df_ior_matched.empty:
        # Find bounds for the transferSize
        if len(df_ior_matched) > 1:
            try:
                row_updates = {'index': index}  # Temporary storage for row-specific updates
                # Determine the step size based on the value of tasksPerNode
                step_tpn = 4 if row['tasksPerNode'] > 40 else 1
                step_t = 8 if row['numTasks'] > 40 else 1

                # Calculate estimated transfer rates for parallelism from 1 to row['tasksPerNode']
                for i in range(0, row['tasksPerNode'] + 1, step_tpn):
                    if i == 0: n = i+1
                    else: n = i
                    par_col = 'tasksPerNode'
                    column_name_ssd = f"estimated_trMiB_ssd_{n}p"
                    estimated_trMiB_ssd = calculate_transferRate_storage(df_ior_matched, n, row['transferSize'], par_col, 'trMiB_ave_ssd')
                    row_updates[column_name_ssd] = estimated_trMiB_ssd

                for i in range(0, row['numTasks'] + 1, step_t):
                    if i == 0: n = i+1
                    else: n = i
                    par_col = 'numTasks'  # Shared storage should use total tasks
                    column_name_beegfs = f"estimated_trMiB_beegfs_{n}p"
                    estimated_trMiB_beegfs = calculate_transferRate_storage(df_ior_matched, n, row['numTasks'], par_col, 'trMiB_ave_beegfs')
                    row_updates[column_name_beegfs] = estimated_trMiB_beegfs

                # Append the row-specific updates to the updates list
                updates.append(row_updates)

            except ValueError as e:
                print(f"Row {index}: Error calculating transferRate - {e}")
        else:
            print(f"Row {index}: Not enough data to determine bounds.")
    else:
        print(f"Row {index}: No matching rows in df_ior.")

# Apply the updates to the dataframe in bulk
for update in updates:
    index = update.pop('index')  # Retrieve the index for the row
    for column, value in update.items():
        wf_pfs_df.at[index, column] = value


    
# Save updated DataFrame to a CSV file
wf_pfs_df.to_csv(f'{test_folders[0]}_tr_estimated.csv', index=False)

# Split dataframe to different task names and save
unique_task_names = wf_pfs_df['taskName'].unique()

for task_name in unique_task_names:
    task_df = wf_pfs_df[wf_pfs_df['taskName'] == task_name].copy()
    
    write_df = task_df[task_df['operation'] == 0]
    read_df = task_df[task_df['operation'] == 1]
    
    # for storage in ['ssd', 'beegfs']:
    #     # Collect dynamically created columns
    #     write_columns = write_df.filter(regex=f'estimated_trMiB_{storage}_\\d+n').columns
    #     read_columns = read_df.filter(regex=f'estimated_trMiB_{storage}_\\d+n').columns
        
    #     # Calculate variances for all dynamically created columns
    #     for col in write_columns:
    #         e_write_var = write_df[col].var()
    #         task_df.loc[task_df['operation'] == 0, f'var_{col}'] = e_write_var
        
    #     for col in read_columns:
    #         e_read_var = read_df[col].var()
    #         task_df.loc[task_df['operation'] == 1, f'var_{col}'] = e_read_var
    
    # # Calculate variance for actual transfer rates (trMiB)
    # a_write_var = write_df['trMiB'].var()
    # a_read_var = read_df['trMiB'].var()
    # task_df.loc[task_df['operation'] == 0, 'var_trMiB'] = a_write_var
    # task_df.loc[task_df['operation'] == 1, 'var_trMiB'] = a_read_var
    
    # Save the task DataFrame to CSV
    task_df.to_csv(f'{test_folders[0]}_{task_name}_tr_estimated.csv', index=False)

ValueError: All arrays must be of the same length

In [ ]:
# Save updated DataFrame to a CSV file
wf_pfs_df.to_csv(f'{test_folders[0]}_tr_estimated.csv', index=False)

# Split dataframe to different task names and save
unique_task_names = wf_pfs_df['taskName'].unique()

for task_name in unique_task_names:
    task_df = wf_pfs_df[wf_pfs_df['taskName'] == task_name].copy()
    
    write_df = task_df[task_df['operation'] == 0]
    read_df = task_df[task_df['operation'] == 1]
    
    # for storage in ['ssd', 'beegfs']:
    #     # Collect dynamically created columns
    #     write_columns = write_df.filter(regex=f'estimated_trMiB_{storage}_\\d+n').columns
    #     read_columns = read_df.filter(regex=f'estimated_trMiB_{storage}_\\d+n').columns
        
    #     # Calculate variances for all dynamically created columns
    #     for col in write_columns:
    #         e_write_var = write_df[col].var()
    #         task_df.loc[task_df['operation'] == 0, f'var_{col}'] = e_write_var
        
    #     for col in read_columns:
    #         e_read_var = read_df[col].var()
    #         task_df.loc[task_df['operation'] == 1, f'var_{col}'] = e_read_var
    
    # # Calculate variance for actual transfer rates (trMiB)
    # a_write_var = write_df['trMiB'].var()
    # a_read_var = read_df['trMiB'].var()
    # task_df.loc[task_df['operation'] == 0, 'var_trMiB'] = a_write_var
    # task_df.loc[task_df['operation'] == 1, 'var_trMiB'] = a_read_var
    
    # Save the task DataFrame to CSV
    task_df.to_csv(f'{test_folders[0]}_{task_name}_tr_estimated.csv', index=False)

In [20]:
# Construct the SPM Here:
pc_df_list = []

task_name_list = list(wf_pfs_df['taskName'].unique())
print(f"Unique task names: {task_name_list}")
# task_name_list.append('none') # for first stage in the workflow

print(wf_pfs_df.columns)
print(wf_pfs_df.shape)
print(wf_pfs_df.head(5))


Unique task names: ['individuals', 'frequency', 'mutation_overlap', 'sifting', 'individuals_merge']
Index(['operation', 'randomOffset', 'transferSize', 'aggregateFilesizeMB',
       'numTasks', 'totalTime', 'numNodes', 'tasksPerNode', 'trMiB',
       'storageType',
       ...
       'estimated_trMiB_ssd_3p', 'estimated_trMiB_ssd_5p',
       'estimated_trMiB_beegfs_2p', 'estimated_trMiB_beegfs_3p',
       'estimated_trMiB_beegfs_4p', 'estimated_trMiB_beegfs_5p',
       'estimated_trMiB_beegfs_6p', 'estimated_trMiB_beegfs_7p',
       'estimated_trMiB_beegfs_9p', 'estimated_trMiB_beegfs_10p'],
      dtype='object', length=103)
(1800, 103)
  operation randomOffset   transferSize  aggregateFilesizeMB  numTasks  \
0         0            0   81791.000000             0.078002       300   
1         1            0    8191.921474          2421.758036       300   
2         1            0    8191.921474          2421.758036       300   
3         1            0    8191.912230          2421.794365

In [21]:
# Identify columns to normalize
estimated_columns = [col for col in wf_pfs_df.columns if col.startswith('estimated_trMiB')]
additional_columns = ['aggregateFilesizeMB', 'opCount']  # Add additional columns for normalization
columns_to_normalize = estimated_columns + additional_columns

# Calculate the global min and max across all selected columns
global_min = wf_pfs_df[columns_to_normalize].min().min()
global_max = wf_pfs_df[columns_to_normalize].max().max()

# Create a dictionary to store normalized values temporarily
normalized_data = {}

# Normalize each column and store in the dictionary
for col in columns_to_normalize:
    norm_col = f"norm_{col}"
    normalized_values = (wf_pfs_df[col] - global_min) / (global_max - global_min)  # Normalize values
    normalized_data[norm_col] = normalized_values + 1  # Shift normalized values by adding 1

# Update the dataframe with the normalized columns in a single operation
for norm_col, values in normalized_data.items():
    wf_pfs_df[norm_col] = values

print(wf_pfs_df.columns)
print(wf_pfs_df.shape)
print(wf_pfs_df.head(6))

Index(['operation', 'randomOffset', 'transferSize', 'aggregateFilesizeMB',
       'numTasks', 'totalTime', 'numNodes', 'tasksPerNode', 'trMiB',
       'storageType',
       ...
       'norm_estimated_trMiB_beegfs_2p', 'norm_estimated_trMiB_beegfs_3p',
       'norm_estimated_trMiB_beegfs_4p', 'norm_estimated_trMiB_beegfs_5p',
       'norm_estimated_trMiB_beegfs_6p', 'norm_estimated_trMiB_beegfs_7p',
       'norm_estimated_trMiB_beegfs_9p', 'norm_estimated_trMiB_beegfs_10p',
       'norm_aggregateFilesizeMB', 'norm_opCount'],
      dtype='object', length=192)
(1800, 192)
  operation randomOffset   transferSize  aggregateFilesizeMB  numTasks  \
0         0            0   81791.000000             0.078002       300   
1         1            0    8191.921474          2421.758036       300   
2         1            0    8191.921474          2421.758036       300   
3         1            0    8191.912230          2421.794365       300   
4         0            0  101056.000000             0.

/tmp/ipykernel_747884/2804968239.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  wf_pfs_df[norm_col] = values
/tmp/ipykernel_747884/2804968239.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  wf_pfs_df[norm_col] = values
/tmp/ipykernel_747884/2804968239.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = f

In [22]:
# Print all rows with taskName == 'sifting'
sifting_rows = wf_pfs_df[wf_pfs_df['prevTask'] == 'sifting']
print(f"wf_pfs_df sifting prevTask rows:\n{sifting_rows['taskName']}")

wf_pfs_df sifting prevTask rows:
16             frequency
27             frequency
32             frequency
36             frequency
42      mutation_overlap
              ...       
1681    mutation_overlap
1746    mutation_overlap
1785    mutation_overlap
1803           frequency
1807           frequency
Name: taskName, Length: 140, dtype: object


In [23]:
def create_empty_df(other_df, taskName, prevTask="none", operation=0):
    MAXTR = 250000  # Maximum transfer rate in MiB/s
    other_df_len = len(other_df)
    
    # Dictionary to hold columns for DataFrame
    df_dict = {
        'operation': [operation] * other_df_len,
        'randomOffset': [0] * other_df_len,
        'transferSize': [1] * other_df_len,
        'aggregateFilesizeMB': [1] * other_df_len,
        'numTasks': [1] * other_df_len,
        'totalTime': [0] * other_df_len,
        'numNodes': [1] * other_df_len,
        'tasksPerNode': [1] * other_df_len,
        'trMiB': [MAXTR] * other_df_len,
        'storageType': ["n/a"] * other_df_len,
        'opCount': [1] * other_df_len,
        'taskName': [taskName] * other_df_len,
        'taskPID': ["none"] * other_df_len,
        'fileName': ["none"] * other_df_len,
        'prevTask': [prevTask] * other_df_len,
        'estimated_trMiB_ssd': [MAXTR] * other_df_len,
        'estimated_trMiB_beegfs': [MAXTR] * other_df_len
    }
    
    # Convert dict to DataFrame
    empty_df = pd.DataFrame(df_dict)
    
    # Debugging output
    print(f"Constructed empty DataFrame:\n{empty_df.head(5)}")
    
    return empty_df




# Build Workflow Graph
Get max stage order for iteration
## First layers Nodes:
- stageOrder == 0
    - Add row as node attributes
    - Need node "order" 

## Middle layers
- Find previous layer nodes  (stageOrder -1)
 - Find if has the sane fileName, and make sure operation is 0 in stageOrder-1 and 1 in stageOrder
 - link edge, calculate IOI and SPM

## Final Layer
- When stageOrder reached max, stops

In [24]:
# Construct Wrokflow Graph from the DataFrame
import networkx as nx

# Sort stage_order_list
stage_order_list = wf_pfs_df['stageOrder'].unique()
stage_order_list = sorted(int(x) for x in stage_order_list)
print(stage_order_list)

# Create the graph
WFG = nx.Graph()
stage_task_node_dict = {}

# Iterate over rows in the filtered DataFrame
for i, row in wf_pfs_df.iterrows():
    nodeName = f"{row['taskName']}:{row['taskPID']}:{row['fileName']}"  # Unique node identifier
    nodeData = row.to_dict()  # Node attributes as dictionary
    stageOrder = row['stageOrder']
    taskName = row['taskName']

    # Add node to graph with its attributes
    WFG.add_node(nodeName, **nodeData)
    if "sifting" in nodeName and "sifted" in nodeName:
        print(f" added node {nodeName}")

    # Populate stage_task_node_dict for faster access
    if stageOrder not in stage_task_node_dict:
        stage_task_node_dict[stageOrder] = {}
    if taskName not in stage_task_node_dict[stageOrder]:
        stage_task_node_dict[stageOrder][taskName] = []

    # Ensure no duplicate nodes
    if nodeName not in stage_task_node_dict[stageOrder][taskName]:
        stage_task_node_dict[stageOrder][taskName].append(nodeName)

print(f"stage_task_node_dict tasks: {stage_task_node_dict.keys()}")

print(f"stage_task_node_dict:")
for k,v in stage_task_node_dict.items():
    print(f"{k}: {v.keys()}")
    for k1, v1 in v.items():
        if k1 == "sifting":
            print(f"{k}: {k1}: {v1}")



# Remove first stage from the list
stage_order_list.pop(0)


[0, 1, 2]
 added node sifting:195109-dc273:sifted.SIFT.chr10.txt
 added node sifting:40333-dc146:sifted.SIFT.chr4.txt
 added node sifting:16492-dc177:sifted.SIFT.chr7.txt
 added node sifting:29050-dc257:sifted.SIFT.chr8.txt
 added node sifting:195581-dc260:sifted.SIFT.chr9.txt
 added node sifting:7294-dc111:sifted.SIFT.chr2.txt
 added node sifting:15919-dc026:sifted.SIFT.chr1.txt
 added node sifting:31942-dc155:sifted.SIFT.chr6.txt
 added node sifting:25632-dc135:sifted.SIFT.chr3.txt
 added node sifting:823-dc148:sifted.SIFT.chr5.txt
stage_task_node_dict tasks: dict_keys([0, 2, 1])
stage_task_node_dict:
0: dict_keys(['individuals'])
2: dict_keys(['frequency', 'mutation_overlap'])
1: dict_keys(['sifting', 'individuals_merge'])
1: sifting: ['sifting:195109-dc273:ALL.chr10.phase3_shapeit2_mvncall_integrated_v5.20130502.sites.annotation.vcf', 'sifting:195109-dc273:sifted.SIFT.chr10.txt', 'sifting:40333-dc146:ALL.chr4.phase3_shapeit2_mvncall_integrated_v5.20130502.sites.annotation.vcf', 'si

0

In [25]:
# Iterate through all nodes in the graph
taskname= "individuals_merge"
print(f"Nodes with prevTask == '{taskname}':")
for node_name, node_data in WFG.nodes(data=True):
    if node_data.get('taskName') == taskname:
        print(f"Node: {node_name}, Data: {node_data}")

Nodes with prevTask == 'individuals_merge':
Node: individuals_merge:195111-dc273:chr10n-3401-3601.tar.gz, Data: {'operation': 1, 'randomOffset': 0, 'transferSize': 1306.1146161934805, 'aggregateFilesizeMB': 2.3691463470458984, 'numTasks': 10, 'totalTime': 0.013046968, 'numNodes': 2, 'tasksPerNode': 5, 'trMiB': 181.58597055238414, 'storageType': 1, 'opCount': 1902, 'taskName': 'individuals_merge', 'taskPID': '195111-dc273', 'fileName': 'chr10n-3401-3601.tar.gz', 'stageOrder': 1, 'prevTask': 'individuals', 'estimated_trMiB_ssd_1p': 1862.7284281791851, 'estimated_trMiB_ssd_4p': 9489.115264093158, 'estimated_trMiB_ssd_8p': nan, 'estimated_trMiB_ssd_12p': nan, 'estimated_trMiB_ssd_16p': nan, 'estimated_trMiB_ssd_20p': nan, 'estimated_trMiB_ssd_24p': nan, 'estimated_trMiB_ssd_28p': nan, 'estimated_trMiB_ssd_32p': nan, 'estimated_trMiB_ssd_36p': nan, 'estimated_trMiB_ssd_40p': nan, 'estimated_trMiB_ssd_44p': nan, 'estimated_trMiB_ssd_48p': nan, 'estimated_trMiB_ssd_52p': nan, 'estimated_trMiB

In [26]:
# Iterate over the remaining stages to create edges between nodes
for currOrder in stage_order_list:
    prevOrder = currOrder - 1

    # Get all nodes in prevOrder and currOrder
    prev_nodes = stage_task_node_dict.get(prevOrder, {})
    curr_nodes = stage_task_node_dict.get(currOrder, {})

    print(f"Processing stageOrder {prevOrder} to {currOrder}")
    print(f"prev_nodes: {list(prev_nodes.keys())}")
    print(f"curr_nodes: {list(curr_nodes.keys())}")

    # Iterate through each task in prevOrder nodes
    for taskName, prevNodeNames in prev_nodes.items():
        for prev_node_name in prevNodeNames:
            prev_fileName = WFG.nodes[prev_node_name]['fileName'].strip()

            # Find current nodes where the prevTask matches the taskName of prev_nodes
            for curr_task_name, currNodeNames in curr_nodes.items():
                for curr_node_name in currNodeNames:
                    prevTask = WFG.nodes[curr_node_name].get('prevTask')
                    if prevTask == taskName:
                        curr_fileName = WFG.nodes[curr_node_name]['fileName'].strip()

                        # Check if the fileName matches to create an edge
                        if prev_fileName == curr_fileName:
                            edge_attributes = {}
                            for storage in ['ssd', 'beegfs']:
                                par_col = 'tasksPerNode'
                                step = 4 if WFG.nodes[prev_node_name][par_col] > 40 else 1
                                if storage == 'beegfs':
                                    par_col = 'numTasks'
                                    step = 8 if WFG.nodes[prev_node_name][par_col] > 40 else 1

                                prev_opCount = WFG.nodes[prev_node_name]['opCount']
                                prev_aggregateFileSize = WFG.nodes[prev_node_name]['aggregateFilesizeMB']
                                curr_opCount = WFG.nodes[curr_node_name]['opCount']
                                curr_aggregateFileSize = WFG.nodes[curr_node_name]['aggregateFilesizeMB']

                                for i in range(0, WFG.nodes[prev_node_name][par_col] + 1, step):
                                    n = i + 1 if i == 0 else i
                                    column_name_prev = f'estimated_trMiB_{storage}_{n}p'
                                    column_name_curr = f'estimated_trMiB_{storage}_{n}p'
                                    prev_task_parallelism = curr_task_parallelism = n

                                    if column_name_prev in WFG.nodes[prev_node_name] and column_name_curr in WFG.nodes[curr_node_name]:
                                        prev_estimated_trMiB = WFG.nodes[prev_node_name].get(column_name_prev)
                                        curr_estimated_trMiB = WFG.nodes[curr_node_name].get(column_name_curr)
                                        
                                    # Handle missing or NaN prev_estimated_trMiB
                                    if math.isnan(prev_estimated_trMiB) or prev_estimated_trMiB is None:
                                        prev_task_parallelism =1
                                        # task_parallelism = max(
                                        #     int(col.split('_')[-1][:-1])
                                        #     for col in WFG.nodes[prev_node_name].keys()
                                        #     if col.startswith(f'norm_estimated_trMiB_{storage}_')
                                        # )
                                        
                                        column_name_prev = f'estimated_trMiB_{storage}_{prev_task_parallelism}p'
                                        prev_estimated_trMiB = WFG.nodes[prev_node_name].get(column_name_prev)
                                        # print(f"prev_node_name[{prev_node_name}] task_parallelism[{task_parallelism}] prev_estimated_trMiB[{prev_estimated_trMiB}]")

                                    # Handle missing or NaN curr_estimated_trMiB
                                    if math.isnan(curr_estimated_trMiB) or curr_estimated_trMiB is None:
                                        curr_task_parallelism =1
                                        # task_parallelism = max(
                                        #     int(col.split('_')[-1][:-1])
                                        #     for col in WFG.nodes[curr_node_name].keys()
                                        #     if col.startswith(f'norm_estimated_trMiB_{storage}_')
                                        # )
                                        
                                        column_name_curr = f'estimated_trMiB_{storage}_{curr_task_parallelism}p'
                                        curr_estimated_trMiB = WFG.nodes[curr_node_name].get(column_name_curr)
                                        # print(f"curr_node_name[{curr_node_name}] task_parallelism[{task_parallelism}] curr_estimated_trMiB[{curr_estimated_trMiB}]")
                                        # print(f"data: {WFG.nodes[curr_node_name]}")
                                    
                                    # # # Calculate estimated task I/O time
                                    # estT_prev = prev_opCount * prev_aggregateFileSize / prev_estimated_trMiB
                                    # estT_curr = curr_opCount * curr_aggregateFileSize / curr_estimated_trMiB
                                    # # Calculate estimated task I/O time
                                    # estT_prev = prev_task_parallelism * prev_opCount * prev_aggregateFileSize / prev_estimated_trMiB
                                    # estT_curr = curr_task_parallelism * prev_opCount * curr_aggregateFileSize / curr_estimated_trMiB
                                                                        # Calculate estimated task I/O time
                                    estT_prev =  prev_aggregateFileSize / (prev_estimated_trMiB)
                                    estT_curr =  curr_aggregateFileSize / (curr_estimated_trMiB)

                                    # Calculate SPM
                                    # SPM = estT_prev / estT_curr if estT_curr > 0 else float('inf')  # Avoid division by zero
                                    SPM = prev_estimated_trMiB / curr_estimated_trMiB if curr_estimated_trMiB > 0 else float('inf')

                                    # # Check for NaN or invalid values
                                    # if math.isnan(SPM) or SPM == float('inf') or SPM == float('-inf'):
                                    #     print(f"Invalid SPM detected for edge ({prev_node_name}, {curr_node_name})")
                                    #     continue  # Skip to the next iteration

                                    # Store attributes only if SPM is valid
                                    edge_attributes[f'estT_prev_{storage}_{n}p'] = estT_prev
                                    edge_attributes[f'estT_curr_{storage}_{n}p'] = estT_curr
                                    edge_attributes[f'SPM_{storage}_{n}p'] = SPM
                                    penalty_factor = abs(SPM - 1)
                                    edge_attributes[f'rank_{storage}_{n}p'] = (estT_prev + estT_curr) * penalty_factor  # Rank based on shifted SPM
                                    # edge_attributes[f'rank_{storage}_{n}p'] = penalty_factor * (estT_prev + estT_curr) # Penalize
                                    
                                    edge_attributes["prod-cons-pair"] = f"{prevTask}:{curr_task_name}"

                            # Add edge to the graph if attributes are valid
                            if edge_attributes:
                                print(f"Adding edge: ({prev_node_name}, {curr_node_name})")
                                WFG.add_edge(prev_node_name, curr_node_name, **edge_attributes)


print(f"Total number of edges in the graph: {WFG.number_of_edges()}")
# Iterate over each edge in the graph along with its attributes
for u, v, attributes in WFG.edges(data=True):
    print(f"Edge from {u} to {v} with attributes: {attributes}")

Processing stageOrder 0 to 1
prev_nodes: ['individuals']
curr_nodes: ['sifting', 'individuals_merge']
Adding edge: (individuals:6818-dc111:chr2n-2201-2401.tar.gz, individuals_merge:7296-dc111:chr2n-2201-2401.tar.gz)
Adding edge: (individuals:25051-dc135:chr3n-1201-1401.tar.gz, individuals_merge:25630-dc135:chr3n-1201-1401.tar.gz)
Adding edge: (individuals:28580-dc257:chr8n-1401-1601.tar.gz, individuals_merge:29049-dc257:chr8n-1401-1601.tar.gz)
Adding edge: (individuals:25170-dc135:chr3n-4601-4801.tar.gz, individuals_merge:25630-dc135:chr3n-4601-4801.tar.gz)
Adding edge: (individuals:194433-dc273:chr10n-5601-5801.tar.gz, individuals_merge:195111-dc273:chr10n-5601-5801.tar.gz)
Adding edge: (individuals:28562-dc257:chr8n-1-201.tar.gz, individuals_merge:29049-dc257:chr8n-1-201.tar.gz)
Adding edge: (individuals:194456-dc273:chr10n-4401-4601.tar.gz, individuals_merge:195111-dc273:chr10n-4401-4601.tar.gz)
Adding edge: (individuals:195099-dc260:chr9n-2201-2401.tar.gz, individuals_merge:195582-

In [27]:
def calculate_SPM_estT_values(WFG):
    """
    Calculate and store weighted SPM values for each producer-consumer pair.
    
    Args:
        WFG (nx.DiGraph): A directed weighted graph with nodes and edges containing performance attributes.
    
    Returns:
        dict: Weighted SPM dictionary containing producer-consumer pairs, their SPM values, and task IO times.
    """
    SPM_estT_values = {}

    # Iterate through all edges in the graph
    for edge in WFG.edges(data=True):
        producer_node, consumer_node, attributes = edge
        prod_cons_pair = f"{WFG.nodes[producer_node]['taskName']}:{WFG.nodes[consumer_node]['taskName']}"
        
        if prod_cons_pair not in SPM_estT_values:
            SPM_estT_values[prod_cons_pair] = {
                'SPM': {},
                'estT_prev': {},
                'estT_curr': {},
                'rank':{},
            }
        
        for key, value in attributes.items():
            if key.startswith("SPM_"):
                storage_n = key.replace("SPM_", "")
                # Append the SPM values
                if storage_n not in SPM_estT_values[prod_cons_pair]['SPM']:
                    SPM_estT_values[prod_cons_pair]['SPM'][storage_n] = []
                SPM_estT_values[prod_cons_pair]['SPM'][storage_n].append(value)
            elif key.startswith("estT_prev_"):
                storage_n = key.replace("estT_prev_", "")
                # Append the estT_prev values
                if storage_n not in SPM_estT_values[prod_cons_pair]['estT_prev']:
                    SPM_estT_values[prod_cons_pair]['estT_prev'][storage_n] = []
                SPM_estT_values[prod_cons_pair]['estT_prev'][storage_n].append(value)
            elif key.startswith("estT_curr_"):
                storage_n = key.replace("estT_curr_", "")
                # Append the estT_curr values
                if storage_n not in SPM_estT_values[prod_cons_pair]['estT_curr']:
                    SPM_estT_values[prod_cons_pair]['estT_curr'][storage_n] = []
                SPM_estT_values[prod_cons_pair]['estT_curr'][storage_n].append(value)
            elif key.startswith("rank_"):
                storage_n = key.replace("rank_", "")
                # Append the estT_curr values
                if storage_n not in SPM_estT_values[prod_cons_pair]['rank']:
                    SPM_estT_values[prod_cons_pair]['rank'][storage_n] = []
                SPM_estT_values[prod_cons_pair]['rank'][storage_n].append(value)
    
    return SPM_estT_values

# Assuming WFG is the networkx graph with edge attributes
SPM_estT_values = calculate_SPM_estT_values(WFG)

# Print weighted SPM values for debugging
for pair, data in SPM_estT_values.items():
    print(f"\nProducer-Consumer Pair: {pair}")
    print("SPM:")
    for storage_n, spm_values in data['SPM'].items():
        print(f"  {storage_n}: {spm_values[0:10]}")
    print("estT_prev:")
    for storage_n, estT_prev_values in data['estT_prev'].items():
        print(f"  {storage_n}: {estT_prev_values[0:10]}")
    print("estT_curr:")
    for storage_n, estT_curr_values in data['estT_curr'].items():
        print(f"  {storage_n}: {estT_curr_values[0:10]}")
    print("rank:")
    for storage_n, estT_curr_values in data['rank'].items():
        print(f"  {storage_n}: {estT_curr_values[0:10]}")



Producer-Consumer Pair: individuals:individuals_merge
SPM:
  ssd_1p: [0.2966460294829862, 0.26410107903515945, 0.24416100221047196, 0.2639587735152907, 0.2965830561575689, 0.24362177786129083, 0.2966071233426747, 0.2372430484600988, 0.23705690409532343, 0.2967189154389379]
  ssd_4p: [0.04300598120981281, 0.04110832642599704, 0.03905317525614587, 0.04080708886497912, 0.042887605424651726, 0.037869454872515806, 0.0429356968736471, 0.03876591404025462, 0.03835179627557414, 0.043151622174869966]
  ssd_8p: [0.3998310348785841, 0.3618891021045343, 0.3308217614346014, 0.3586739419626615, 0.3984905017939534, 0.31863887045037786, 0.3990342604222243, 0.3246360923521013, 0.3204304653123922, 0.4014777736747623]
  ssd_12p: [0.5330839997806585, 0.4708096812352224, 0.4376582497985651, 0.4724883086619219, 0.533774207986893, 0.44401890806757927, 0.5334903129065103, 0.4232183273157544, 0.4254140751428634, 0.5322242413904388]
  ssd_16p: [0.23645889485141433, 0.21713858712908554, 0.1965588864737447, 0.21

In [28]:
# Function to calculate averages for SPM_estT_values
def calculate_averages(SPM_estT_values):
    ave_SPM_estT_values = {}

    for pair, data in SPM_estT_values.items():
        ave_SPM_estT_values[pair] = {
            'SPM': {},
            'estT_prev': {},
            'estT_curr': {},
            'rank':{},
        }

        # Average the SPM values
        for storage_n, spm_values in data['SPM'].items():
            ave_SPM_estT_values[pair]['SPM'][storage_n] = np.mean(spm_values)

        # Average the estT_prev values
        for storage_n, estT_prev_values in data['estT_prev'].items():
            ave_SPM_estT_values[pair]['estT_prev'][storage_n] = np.mean(estT_prev_values)

        # Average the estT_curr values
        for storage_n, estT_curr_values in data['estT_curr'].items():
            ave_SPM_estT_values[pair]['estT_curr'][storage_n] = np.mean(estT_curr_values)
            
        # Average the rank values
        for storage_n, rank_val in data['rank'].items():
            ave_SPM_estT_values[pair]['rank'][storage_n] = np.mean(rank_val)

    return ave_SPM_estT_values


# Calculate averages
ave_SPM_estT_values = calculate_averages(SPM_estT_values)

# Print the averaged results
for pair, data in ave_SPM_estT_values.items():
    print(f"\nProducer-Consumer Pair: {pair}")
    print("Averaged SPM:")
    for storage_n, avg_spm in data['SPM'].items():
        print(f"  {storage_n}: {avg_spm}")
    print("Averaged estT_prev:")
    for storage_n, avg_estT_prev in data['estT_prev'].items():
        print(f"  {storage_n}: {avg_estT_prev}")
    print("Averaged estT_curr:")
    for storage_n, avg_estT_curr in data['estT_curr'].items():
        print(f"  {storage_n}: {avg_estT_curr}")
    print("Averaged rank:")
    for storage_n, ave_rank in data['rank'].items():
        print(f"  {storage_n}: {ave_rank}")


Producer-Consumer Pair: individuals:individuals_merge
Averaged SPM:
  ssd_1p: 0.2639499439511424
  ssd_4p: 0.040598014921734406
  ssd_8p: 0.3573324638994695
  ssd_12p: 0.47332311087266676
  ssd_16p: 0.21215216106708498
  ssd_20p: 0.38872326336149515
  ssd_24p: 0.38872326336149515
  ssd_28p: 0.38872326336149515
  ssd_32p: 0.35303471701076333
  ssd_36p: 0.4551336922632025
  ssd_40p: 0.4551336922632025
  ssd_44p: 0.4551336922632025
  ssd_48p: 0.4551336922632025
  ssd_52p: 0.4551336922632025
  ssd_56p: 0.4551336922632025
  ssd_60p: 0.4551336922632025
  ssd_64p: 0.5002157216630982
  ssd_68p: 0.20651627091611122
  ssd_72p: 0.20651627091611122
  ssd_76p: 0.20651627091611122
  ssd_80p: 0.20651627091611122
  ssd_84p: 0.20651627091611122
  ssd_88p: 0.20651627091611122
  ssd_92p: 0.20651627091611122
  ssd_96p: 0.20651627091611122
  ssd_100p: 0.20651627091611122
  ssd_104p: 0.20651627091611122
  ssd_108p: 0.20651627091611122
  ssd_112p: 0.20651627091611122
  ssd_116p: 0.20651627091611122
  ssd_12

In [29]:
def display_top_sorted_averaged_rank(ave_SPM_estT_values, baseline=0, top_n=5):
    print(f"Top {top_n} Averaged SPM Values Closest to Baseline = {baseline}:\n")

    for pair, data in ave_SPM_estT_values.items():
        # Reverse the key display order if needed
        producer, consumer = pair.split(":")
        print(f"Producer: {producer}, Consumer: {consumer}")
        
        # Collect and sort SPM values by closeness to the baseline
        sorted_spm = sorted(data['rank'].items(), key=lambda item: abs(item[1] - baseline))
        
        # Display the top N SPM values
        for rank, (storage_n, avg_spm) in enumerate(sorted_spm[:top_n], start=1):
            print(f"  Rank {rank}: {storage_n} with Averaged SPM = {avg_spm}")
        print()  # Blank line for readability


# Call the function to display results
display_top_sorted_averaged_rank(ave_SPM_estT_values, top_n=5)

Top 5 Averaged SPM Values Closest to Baseline = 0:

Producer: individuals, Consumer: individuals_merge
  Rank 1: beegfs_8p with Averaged SPM = 0.0
  Rank 2: ssd_4p with Averaged SPM = 0.00043128412049854564
  Rank 3: ssd_64p with Averaged SPM = 0.0006299213282158081
  Rank 4: ssd_12p with Averaged SPM = 0.000666433499056349
  Rank 5: ssd_36p with Averaged SPM = 0.0006913891229423508

Producer: frequency, Consumer: individuals_merge
  Rank 1: beegfs_1p with Averaged SPM = 0.0
  Rank 2: beegfs_2p with Averaged SPM = 0.0
  Rank 3: beegfs_3p with Averaged SPM = 0.0
  Rank 4: beegfs_4p with Averaged SPM = 0.0
  Rank 5: beegfs_5p with Averaged SPM = 0.0

Producer: frequency, Consumer: sifting
  Rank 1: beegfs_1p with Averaged SPM = 0.0
  Rank 2: beegfs_2p with Averaged SPM = 0.0
  Rank 3: beegfs_3p with Averaged SPM = 0.0
  Rank 4: beegfs_4p with Averaged SPM = 0.0
  Rank 5: beegfs_5p with Averaged SPM = 0.0

Producer: mutation_overlap, Consumer: individuals_merge
  Rank 1: beegfs_1p with Av

In [30]:
# Initialize SPM values dictionary with extended storage
SPM_values_dict = {}
print(f"task_name_list: {task_name_list}")

# Option to calculate from sum or average
use_average = True  # Set to False to use sum instead of average

# Get all producer-consumer pairs
for task in task_name_list:
    # Get all nodes with attribute taskName == task
    target_nodes = [node for node, data in WFG.nodes(data=True) if data.get('taskName') == task]

    # For each target node, find all consumer nodes and calculate SPM values
    for producer_node in target_nodes:
        # Retrieve SPM values for all storage types and consumer connections
        for consumer_node in WFG.neighbors(producer_node):
            # Ensure that we are analyzing producer-to-consumer direction
            consumer_prevTask_name = str(WFG.nodes[consumer_node].get('prevTask')).split(":")[0]
            if consumer_prevTask_name == task:
                # Initialize dictionary to store SPM, estT_prev, and estT_curr lists
                pair_key = f"{WFG.nodes[producer_node]['taskName']}:{WFG.nodes[consumer_node]['taskName']}"
                if pair_key not in SPM_values_dict:
                    SPM_values_dict[pair_key] = {
                        "SPM": {}, 
                        "estT_prev": {}, 
                        "estT_curr": {}, 
                        "averages": {}  # New section for averaged values
                    }
                
                # Retrieve the edge data between producer and consumer
                if WFG.has_edge(producer_node, consumer_node):
                    edge_data = WFG.get_edge_data(producer_node, consumer_node)

                    # Process SPM for each storage type
                    for storage in ['ssd', 'beegfs']:
                        par_col = 'tasksPerNode'
                        tpn = WFG.nodes[producer_node][par_col]
                        # Determine the step size based on the value of tasksPerNode
                        step = 4 if tpn > 40 else 1
                        if storage == 'beegfs': 
                            par_col = 'numTasks'
                            tpn = WFG.nodes[producer_node][par_col]
                            step = 8 if tpn > 40 else 1

                        max_parallelism = WFG.nodes[producer_node].get(f'tasksPerNode_{storage}', WFG.nodes[producer_node][par_col])
                        for i in range(0, max_parallelism + 1, step):
                            if i == 0: n = i+1
                            else: n = i
                            spm_key = f"SPM_{storage}_{n}p"
                            estT_prev_key = f"estT_prev_{storage}_{n}p"
                            estT_curr_key = f"estT_curr_{storage}_{n}p"

                            # Initialize lists if not already present
                            if spm_key not in SPM_values_dict[pair_key]["SPM"]:
                                SPM_values_dict[pair_key]["SPM"][spm_key] = []
                                SPM_values_dict[pair_key]["estT_prev"][estT_prev_key] = []
                                SPM_values_dict[pair_key]["estT_curr"][estT_curr_key] = []

                            # Add values to the lists
                            if spm_key in edge_data and estT_prev_key in edge_data and estT_curr_key in edge_data:
                                spm_value = edge_data[spm_key]
                                estT_prev_value = edge_data[estT_prev_key]
                                estT_curr_value = edge_data[estT_curr_key]

                                SPM_values_dict[pair_key]["SPM"][spm_key].append(spm_value)
                                SPM_values_dict[pair_key]["estT_prev"][estT_prev_key].append(estT_prev_value)
                                SPM_values_dict[pair_key]["estT_curr"][estT_curr_key].append(estT_curr_value)
                            else:
                                print(f"Missing data for edge {producer_node}-{consumer_node}, key: {spm_key}")


# Calculate averaged values for each {storage}_{n}p
for pair_key, data in SPM_values_dict.items():
    for category, values in data.items():
        if category not in ["SPM", "estT_prev", "estT_curr"]:
            continue
        for key, value_list in values.items():
            avg_key = f"avg_{key}"
            if len(value_list) > 0:
                average_value = sum(value_list) / len(value_list)
            else:
                average_value = 0
            SPM_values_dict[pair_key]["averages"][avg_key] = average_value

# Output the final extended SPM dictionary
print("Final SPM Values Dictionary with Averages:")
for pair_key, data in SPM_values_dict.items():
    print(f"\nProducer-Consumer Pair: {pair_key}")
    print("Averaged Values:")
    for avg_key, avg_value in data["averages"].items():
        print(f"  {avg_key}: {avg_value}")

task_name_list: ['individuals', 'frequency', 'mutation_overlap', 'sifting', 'individuals_merge']
Final SPM Values Dictionary with Averages:

Producer-Consumer Pair: individuals:individuals_merge
Averaged Values:
  avg_SPM_ssd_1p: 0.2613645236250812
  avg_SPM_ssd_4p: 0.04038738371081688
  avg_SPM_ssd_8p: 0.3538990745598758
  avg_SPM_ssd_12p: 0.46864416603321113
  avg_SPM_ssd_16p: 0.2101486934678046
  avg_SPM_ssd_20p: 0.38490239956146566
  avg_SPM_ssd_24p: 0.38490239956146566
  avg_SPM_ssd_28p: 0.38490239956146566
  avg_SPM_ssd_32p: 0.34963084420703455
  avg_SPM_ssd_36p: 0.45070549200255905
  avg_SPM_ssd_40p: 0.45070549200255905
  avg_SPM_ssd_44p: 0.45070549200255905
  avg_SPM_ssd_48p: 0.45070549200255905
  avg_SPM_ssd_52p: 0.45070549200255905
  avg_SPM_ssd_56p: 0.45070549200255905
  avg_SPM_ssd_60p: 0.45070549200255905
  avg_SPM_ssd_64p: 0.49528077814646776
  avg_SPM_ssd_68p: 0.20456494846320744
  avg_SPM_ssd_72p: 0.20456494846320744
  avg_SPM_ssd_76p: 0.20456494846320744
  avg_SPM_ssd_

In [31]:
# Function to display the top n ranked {store}_{n}p configurations
def display_top_n_configurations(SPM_values_dict, n=10):
    # Baseline value
    baseline = 1
    for pair_key, data in SPM_values_dict.items():
        print(f"\nProducer-Consumer Pair: {pair_key}")
        print(f"Top {n} Ranked {storage}_{n}p Configurations (Closest to Baseline={baseline}):")

        # Collect all {store}_{n}p configurations with their average SPM values
        all_spm_configs = []
        for spm_key, spm_values in data.get("SPM", {}).items():
            if isinstance(spm_values, list) and len(spm_values) > 0:
                # Calculate the average SPM for each configuration
                avg_spm = sum(spm_values) / len(spm_values)
                all_spm_configs.append((spm_key, avg_spm))

        # Sort configurations by absolute difference from the baseline
        sorted_configs = sorted(all_spm_configs, key=lambda x: abs(x[1] - baseline))

        # Display the top n configurations
        for rank, (spm_key, avg_spm) in enumerate(sorted_configs[:n], start=1):
            print(f"  Rank {rank}: {spm_key} with Average SPM = {avg_spm}")

# Display the results
display_top_n_configurations(SPM_values_dict)


Producer-Consumer Pair: individuals:individuals_merge
Top 10 Ranked beegfs_10p Configurations (Closest to Baseline=1):
  Rank 1: SPM_beegfs_8p with Average SPM = 1.0
  Rank 2: SPM_ssd_64p with Average SPM = 0.49528077814646776
  Rank 3: SPM_ssd_12p with Average SPM = 0.46864416603321113
  Rank 4: SPM_ssd_36p with Average SPM = 0.45070549200255905
  Rank 5: SPM_ssd_40p with Average SPM = 0.45070549200255905
  Rank 6: SPM_ssd_44p with Average SPM = 0.45070549200255905
  Rank 7: SPM_ssd_48p with Average SPM = 0.45070549200255905
  Rank 8: SPM_ssd_52p with Average SPM = 0.45070549200255905
  Rank 9: SPM_ssd_56p with Average SPM = 0.45070549200255905
  Rank 10: SPM_ssd_60p with Average SPM = 0.45070549200255905

Producer-Consumer Pair: sifting:frequency
Top 10 Ranked beegfs_10p Configurations (Closest to Baseline=1):
  Rank 1: SPM_beegfs_1p with Average SPM = 1.0
  Rank 2: SPM_beegfs_2p with Average SPM = 1.0
  Rank 3: SPM_beegfs_3p with Average SPM = 1.0
  Rank 4: SPM_beegfs_4p with Avera

In [32]:
def calculate_average_SPM_by_storage(SPM_values_dict):
    """
    Calculate the average SPM across all parallelism levels for each storage type 
    for each producer-consumer pair.

    Args:
        SPM_values_dict (dict): Dictionary containing SPM values for producer-consumer pairs.

    Returns:
        dict: A dictionary with average SPM for each storage type for each pair.
    """
    average_SPM_by_storage = {}

    for pair_key, data in SPM_values_dict.items():
        average_SPM_by_storage[pair_key] = {}
        
        for spm_key, spm_values in data.get("SPM", {}).items():
            # Extract storage type from the key (e.g., ssd or beegfs)
            storage_type = spm_key.split("_")[1]
            
            # Initialize the storage entry if not already present
            if storage_type not in average_SPM_by_storage[pair_key]:
                average_SPM_by_storage[pair_key][storage_type] = []

            # Add the SPM values to the storage type
            if isinstance(spm_values, list) and len(spm_values) > 0:
                average_SPM_by_storage[pair_key][storage_type].extend(spm_values)

    # Calculate the average for each storage type
    for pair_key, storage_data in average_SPM_by_storage.items():
        for storage_type, spm_values in storage_data.items():
            if len(spm_values) > 0:
                average_SPM_by_storage[pair_key][storage_type] = sum(spm_values) / len(spm_values)
            else:
                average_SPM_by_storage[pair_key][storage_type] = None

    return average_SPM_by_storage


def display_average_SPM_by_storage(average_SPM_by_storage):
    """
    Display the average SPM for each storage type for each producer-consumer pair
    and suggest the storage type closer to baseline=1.

    Args:
        average_SPM_by_storage (dict): Dictionary with average SPM values by storage type.
    """
    baseline = 1  # Define the baseline value
    print("\nAverage SPM by Storage for Each Producer-Consumer Pair:")

    for pair_key, storage_data in average_SPM_by_storage.items():
        print(f"\nProducer-Consumer Pair: {pair_key}")

        # Display each storage type and calculate the difference from baseline
        closest_storage = None
        closest_diff = float('inf')

        for storage_type, avg_spm in storage_data.items():
            if avg_spm is not None:
                diff = abs(avg_spm - baseline)
                print(f"  {storage_type}: Average SPM = {avg_spm:.6f} (Difference from baseline = {diff:.6f})")

                # Update the closest storage if this one is closer to the baseline
                if diff < closest_diff:
                    closest_diff = diff
                    closest_storage = storage_type
            else:
                print(f"  {storage_type}: No data available")

        # Print the suggestion for the storage closer to the baseline
        if closest_storage:
            print(f"  Suggested Storage: {closest_storage}") # (Closest to baseline with Difference = {closest_diff:.6f})
        else:
            print("  No valid storage suggestions available.")

# Calculate the average SPM by storage
average_SPM_by_storage = calculate_average_SPM_by_storage(SPM_values_dict)

# Display the results
display_average_SPM_by_storage(average_SPM_by_storage)



Average SPM by Storage for Each Producer-Consumer Pair:

Producer-Consumer Pair: individuals:individuals_merge
  ssd: Average SPM = 0.282514 (Difference from baseline = 0.717486)
  beegfs: Average SPM = 87.007421 (Difference from baseline = 86.007421)
  Suggested Storage: ssd

Producer-Consumer Pair: sifting:frequency
  ssd: Average SPM = 0.248656 (Difference from baseline = 0.751344)
  beegfs: Average SPM = 1.000000 (Difference from baseline = 0.000000)
  Suggested Storage: beegfs

Producer-Consumer Pair: sifting:mutation_overlap
  ssd: Average SPM = 0.102591 (Difference from baseline = 0.897409)
  beegfs: Average SPM = 1.000000 (Difference from baseline = 0.000000)
  Suggested Storage: beegfs

Producer-Consumer Pair: individuals_merge:frequency
  ssd: Average SPM = 0.251565 (Difference from baseline = 0.748435)
  beegfs: Average SPM = 1.000000 (Difference from baseline = 0.000000)
  Suggested Storage: beegfs

Producer-Consumer Pair: individuals_merge:mutation_overlap
  ssd: Average 

In [33]:
def find_best_storage_and_parallelism(SPM_values_dict):
    """
    Find the best storage and its parallelism configuration based on SPM closest to baseline=1.

    Args:
        SPM_values_dict (dict): Dictionary containing SPM values.

    Returns:
        None: Prints the results.
    """
    baseline = 1
    best_storage = None
    best_storage_avg_spm = float('inf')
    storage_averages = {}

    # Calculate averaged SPM for each storage across all producer-consumer pairs
    for pair_key, data in SPM_values_dict.items():
        print(f"\nProducer-Consumer Pair: {pair_key}")

        # Aggregate SPM values for all storage types
        storage_spm_values = {}
        for spm_key, spm_values in data.get("SPM", {}).items():
            if len(spm_values) > 0:
                avg_spm = sum(spm_values) / len(spm_values)
                storage_type = spm_key.split("_")[0]  # Extract storage type (e.g., 'ssd', 'beegfs')
                if storage_type not in storage_spm_values:
                    storage_spm_values[storage_type] = []
                storage_spm_values[storage_type].append((spm_key, avg_spm))

        # Select the best storage and its configuration
        for storage, spm_list in storage_spm_values.items():
            avg_spm_for_storage = sum(spm[1] for spm in spm_list) / len(spm_list)
            storage_averages[storage] = avg_spm_for_storage

            if abs(avg_spm_for_storage - baseline) < abs(best_storage_avg_spm - baseline):
                best_storage = storage
                best_storage_avg_spm = avg_spm_for_storage

    # Display the best storage
    print("\nBest Selected Storage:")
    if best_storage:
        print(f"  {best_storage} with Average SPM = {best_storage_avg_spm}")

    # Display the best parallelism configuration for the selected storage
    print("\nBest Parallelism Configuration for the Selected Storage:")
    for pair_key, data in SPM_values_dict.items():
        if best_storage:
            best_config = None
            best_config_spm = float('inf')

            for spm_key, spm_values in data.get("SPM", {}).items():
                if spm_key.startswith(best_storage) and len(spm_values) > 0:
                    avg_spm = sum(spm_values) / len(spm_values)
                    if abs(avg_spm - baseline) < abs(best_config_spm - baseline):
                        best_config = spm_key
                        best_config_spm = avg_spm

            if best_config:
                print(f"  {pair_key}: {best_config} with Average SPM = {best_config_spm}")

# Example usage
find_best_storage_and_parallelism(SPM_values_dict)



Producer-Consumer Pair: individuals:individuals_merge

Producer-Consumer Pair: sifting:frequency

Producer-Consumer Pair: sifting:mutation_overlap

Producer-Consumer Pair: individuals_merge:frequency

Producer-Consumer Pair: individuals_merge:mutation_overlap

Best Selected Storage:
  SPM with Average SPM = 0.7464352197905543

Best Parallelism Configuration for the Selected Storage:
  individuals:individuals_merge: SPM_beegfs_8p with Average SPM = 1.0
  sifting:frequency: SPM_beegfs_1p with Average SPM = 1.0
  sifting:mutation_overlap: SPM_beegfs_1p with Average SPM = 1.0
  individuals_merge:frequency: SPM_beegfs_1p with Average SPM = 1.0
  individuals_merge:mutation_overlap: SPM_beegfs_1p with Average SPM = 1.0
